# Exploración del modelo
## Aprendizaje Profundo - PCIC
### Eduardo García Alarcón 2025-1

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import speechbrain as sb
import torch as th
import torch.nn as nn
import torchaudio.transforms as T
from tqdm.auto import trange
import torchaudio
import pandas as pd
import os
import librosa


SEED = 42
np.random.seed(SEED)
th.manual_seed(SEED)

DEVICE = 'cuda:0' if th.cuda.is_available() else 'cpu'
print(f'{DEVICE = }')

In [ ]:
from speechbrain.inference.classifiers import EncoderClassifier
classifier = EncoderClassifier.from_hparams(source="speechbrain/urbansound8k_ecapa", savedir="pretrained_models/gurbansound8k_ecapa")

In [ ]:
classifier.modules

## Clases relevantes

### Clases de UrbanSound
- 0 = dog_bark
- 1 = children_playing
- 2 = air_conditioner
- 3 = street_music
- 4 = gun_shot
- 5 = siren
- 6 = engine_idling
- 7 = jackhammer
- 8 = drilling
- 9 = car_horn

In [ ]:
lable_to_code = {
  'Bark': '/m/05tny_', 
  'Children playing': '/t/dd00013',
  'Air conditioning': '/m/025wky1',
  # 'Music',
  'Gunshot, gunfire': '/m/032s66',
  'Police car (siren)': '/m/04qvtq', 'Ambulance (siren)': '/m/012n7d',
  'Engine': '/m/02mk9',
  'Jackhammer': '/m/03p19w',
  'Drill': '/m/01d380',
  'Vehicle horn, car horn, honking': '/m/0912c9',
}

In [ ]:
code_to_lable  = {item: key for key, item in lable_to_code.items()}

In [ ]:
code_to_lable

In [ ]:
code_to_indx = {key: i for i, key in enumerate(code_to_lable.keys())}

In [ ]:
indx_to_lable = {}
for code, indx in code_to_indx.items():
    indx_to_lable[indx] = code_to_lable[code]

In [ ]:
indx_to_lable

In [ ]:
code_to_indx.items()

In [ ]:
code_to_indx

In [ ]:
codes = [key for key in code_to_lable.keys()]

In [ ]:
codes

## Dataframes

### Abrimos los CSV

In [ ]:
df_val = pd.read_csv(
  'http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/eval_segments.csv',
	delimiter=', ',
	header=2)
    
df_train = pd.read_csv(
  'http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/balanced_train_segments.csv',
  delimiter=', ',
	header=2)
    
df_train_unbalanced = pd.read_csv(
  'http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/unbalanced_train_segments.csv',
  delimiter=', ',
	header=2)

In [ ]:
df_train.head()

In [ ]:
df_train_unbalanced.head()

In [ ]:
df_val.head()

---------

df_train['valid'] = df_train['positive_labels'].apply(lambda x: any(code in x for code in codes) if x else False)

df_train_unbalanced['valid'] = df_train_unbalanced['positive_labels'].apply(lambda x: any(code in x for code in codes) if x else False)

df_val['valid'] = df_val['positive_labels'].apply(lambda x: any(code in x for code in codes) if x else False)

df_train[df_train.valid].valid.sum()

df_train_unbalanced[df_train_unbalanced.valid].valid.sum()

df_val[df_val.valid].valid.sum()

------

## Descargamos los datos

In [ ]:
from Util import audioSet_download, file_mover, actual_df

In [ ]:
audioSet_download(lable_to_code, dataset='unbalanced_train')

In [ ]:
archivos_disponibles = file_mover()


In [ ]:

df = actual_df(df_train_unbalanced.copy(), archivos_disponibles)


## Usaremos los datos de unbalanced y extraeremos solo las 10 clases relevantes
Y haremos el split de los datos para con 80% entrenamiento y 20% entrenamiento 

In [ ]:
df

#### Creamos una columna para las categorías

In [ ]:
df['indx'] = df['positive_labels'].apply(lambda x: next((code_to_indx.get(code) for code in codes if code in x), None) if pd.notnull(x) else None)

In [ ]:
df

In [ ]:
idx, cuentas = np.unique(df.indx, return_counts=True)
plt.bar(x=[indx_to_lable[indx] for indx in idx], height=cuentas)
plt.xticks(rotation=90)
plt.xlabel('Categoría')
plt.ylabel('Número de audios')
plt.show()

### Hacemos el split de datos

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df_train, df_val = train_test_split(df, test_size=.2, random_state=SEED, shuffle=True, stratify=df.indx)

In [ ]:
df_train.shape

In [ ]:
df_val.shape

In [ ]:
idx_ent, cuentas_ent = np.unique(df_train.indx, return_counts=True)
idx_val, cuentas_val = np.unique(df_val.indx, return_counts=True)

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5))

axs[0].bar(x=[indx_to_lable[indx] for indx in idx_ent], height=cuentas_ent)
axs[0].set_title('Entrenamiento')
axs[0].set_xlabel('Categoría')
axs[0].set_ylabel('Número de imágenes')
axs[0].tick_params(rotation=87)

axs[1].bar(x=[indx_to_lable[indx] for indx in idx_val], height=cuentas_val)
axs[1].set_title('Validación')
axs[1].set_xlabel('Categoría')
axs[1].set_ylabel('Número de imágenes')
axs[1].tick_params(rotation=87)

plt.show()

### Data Augmentation

In [ ]:
class AudioPadding(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, waveform):
    expected_size = 10 * 16000 # tiempo * sample rate
    if waveform.shape[1] < expected_size:
      padding_len = expected_size - len(waveform)
      padding = th.zeros(1, padding_len)
      waveform = th.cat([waveform, padding], 1)

    return waveform

In [ ]:
class RandomSilence(nn.Module):
  def __init__(self, ratio=.3):
    super().__init__()
    self.ratio = ratio

  def forward(self, waveform):
    samples = len(waveform)
    # print(samples)
    silence_size = int(samples * self.ratio)
    # print(f'{silence_size = }')
    start = np.random.randint(0, samples - silence_size)
    # print(f'{start = }')
    waveform[:, start:start+silence_size] = 0.0
    # if waveform.shape
    return waveform

In [ ]:
class SizeChecker(nn.Module):
  def __init__(self):
    super(SizeChecker, self).__init__()

  def forward(self, mfcc):
    if mfcc.size(2) > 800:
      mfcc = mfcc[:, :, :800]
    return mfcc
    

In [ ]:
tr_train = nn.Sequential(
  AudioPadding(),
  RandomSilence(),
  T.MFCC(sample_rate=16000, n_mfcc=80),
  SizeChecker()
)

In [ ]:
tr_val = nn.Sequential(
  AudioPadding(),
  # RandomSilence(),
  T.MFCC(sample_rate=16000, n_mfcc=80),
  SizeChecker()
)

### Dataset y  DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class AudioLoader(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.root = './Data/Audios/'

        
    def __getitem__(self, idx):
        cat = self.df.indx.iloc[idx] # Categoría
        # Columna
        tmp_row = self.df.iloc[idx]
        # Generamos el nombre del archivo
        name = f"{tmp_row.iloc[0]}_{tmp_row.iloc[1]}-{tmp_row.iloc[2]}."
        # print(f'{name=}')/
        # Buscamos la extensión
        for filename in os.listdir(self.root):
            if filename.startswith(name):
                name = filename
                # print(f'{filename = }')
                
        # Abrimos el audio
        audio, sr = librosa.load(os.path.join(self.root, name), sr=16000)
        # print(f'{len(audio) = }')
        audio = th.tensor(audio[np.newaxis, :])
        
        if self.transform:
            x = self.transform(audio)
            
        return x, th.tensor(cat, dtype=th.long)


    def __len__(self):
        return len(self.df) 

In [ ]:
ds_train = AudioLoader(df_train, transform=tr_train)
ds_val   = AudioLoader(df_val, transform=tr_val)

In [ ]:
it_train = iter(ds_train)

In [ ]:
a = next(it_train)

In [ ]:
dl_train = DataLoader(ds_train, batch_size=64, shuffle=True)
dl_val   = DataLoader(ds_val, batch_size=64)

In [ ]:
# desplegamos un lote de ejemplos
x, labels = next(iter(dl_train))
print(f'x shape={x.shape} dtype={x.dtype}')
# print(f'y shape={y.shape} dtype={y.dtype}')
print(f'{labels = }')